In [1]:
from ml_end_to_end.features import build_features
df = build_features()
print(df.shape)

(3000888, 16)


In [2]:
from sklearn.model_selection import TimeSeriesSplit
import pandas as pd

df = df.sort_values("date").reset_index(drop=True)
dates = df["date"]

tscv = TimeSeriesSplit(n_splits=4, test_size=len(df)//10, gap=0)
for i, (tr, te) in enumerate(tscv.split(df)):
    print(i, dates.iloc[tr].min().date(), dates.iloc[tr].max().date(), "|",
          dates.iloc[te].min().date(), dates.iloc[te].max().date())

0 2013-01-01 2015-10-10 | 2015-10-10 2016-03-27
1 2013-01-01 2016-03-27 | 2016-03-27 2016-09-12
2 2013-01-01 2016-09-12 | 2016-09-12 2017-02-28
3 2013-01-01 2017-02-28 | 2017-02-28 2017-08-15


In [3]:
tscv_gap = TimeSeriesSplit(n_splits=4, test_size=len(df)//10, gap=len(df)//50)
for i, (tr, te) in enumerate(tscv_gap.split(df)):
    print(i, dates.iloc[tr].max().date(), "->", dates.iloc[te].min().date())

0 2015-09-06 -> 2015-10-10
1 2016-02-23 -> 2016-03-27
2 2016-08-09 -> 2016-09-12
3 2017-01-25 -> 2017-02-28


In [4]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=4)
for i, (tr, te) in enumerate(gkf.split(df, groups=df["store_nbr"])):
    tr_stores = set(df["store_nbr"].iloc[tr])
    te_stores = set(df["store_nbr"].iloc[te])
    print(i, len(tr_stores), len(te_stores), "ortak mağaza:", len(tr_stores & te_stores))

0 40 14 ortak mağaza: 0
1 40 14 ortak mağaza: 0
2 41 13 ortak mağaza: 0
3 41 13 ortak mağaza: 0


In [5]:
cut = pd.Timestamp("2016-09-12")
train = df[df["date"] <= cut]
valid = df[df["date"] > cut]

for col in ["sales", "onpromotion", "log_sales"]:
    print(col, round(train[col].mean(), 3), round(valid[col].mean(), 3))

print("train ortalama onpromotion>0 oranı:", (train["onpromotion"] > 0).mean().round(3))
print("valid ortalama onpromotion>0 oranı:", (valid["onpromotion"] > 0).mean().round(3))

sales 328.287 476.081
onpromotion 1.47 7.149
log_sales 2.775 3.532
train ortalama onpromotion>0 oranı: 0.144
valid ortalama onpromotion>0 oranı: 0.444


In [ ]:
df.groupby(df["date"].dt.year)["onpromotion"].apply(lambda s: (s > 0).mean()).round(3)